# STRAT-002 Robustness Test: SOPR + Realized Loss

**Initial Finding:** SOPR + RL Z > 0.5 achieved 67% beat rate.

**Question:** Is this robust or overfit to one parameter?

**Test:** Grid search across RL Z thresholds × MVRV exit params

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("SOPR + Realized Loss Robustness Test 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")

df = realized_loss.join(price, how='inner').join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner')
df = df.sort_index()

# Create RL z-score
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

df_test = df[df.index >= '2018-12-15'].copy()
df_test = df_test.dropna(subset=['rl_zscore'])

print(f"Data: {len(df_test)} rows")
print(f"Date range: {df_test.index.min().date()} to {df_test.index.max().date()}")

In [ ]:
def combined_entry(df, rl_z_threshold):
    """Entry when SOPR double cap AND realized loss spike."""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    rl_signal = df['rl_zscore'] > rl_z_threshold
    combined = sopr_signal & rl_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

def backtest_mvrv_trailing(df, entries, mvrv_trigger=2.25, trailing_pct=0.20, stop_loss=0.20, max_hold_days=365):
    """Backtest with MVRV trailing exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({'pnl_pct': pnl, 'exit_reason': exit_reason})
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
def walk_forward(df, rl_z_threshold, mvrv_trigger, trailing_pct):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    total_signals = 0
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = combined_entry(test_df, rl_z_threshold)
        total_signals += entries.sum()
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_df = pd.DataFrame(results)
    beat_rate = wf_df['beat_hold'].mean()
    avg_excess = (wf_df['strat_return'] - wf_df['hold_return']).mean()
    
    return beat_rate, avg_excess, total_signals

---
## Grid Search

In [ ]:
# Grid parameters
rl_z_thresholds = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
mvrv_triggers = [2.0, 2.25, 2.5, 2.75, 3.0]
trailing_pcts = [0.15, 0.20, 0.25, 0.30]

total_combos = len(rl_z_thresholds) * len(mvrv_triggers) * len(trailing_pcts)
print(f"Testing {total_combos} parameter combinations...")

grid_results = []

for rl_z in rl_z_thresholds:
    for mvrv_t in mvrv_triggers:
        for trail in trailing_pcts:
            beat_rate, avg_excess, signals = walk_forward(df_test, rl_z, mvrv_t, trail)
            
            grid_results.append({
                'rl_z': rl_z,
                'mvrv_trigger': mvrv_t,
                'trailing_pct': trail,
                'beat_rate': beat_rate,
                'avg_excess': avg_excess,
                'signals': signals
            })

grid_df = pd.DataFrame(grid_results)
print(f"\nCompleted {len(grid_df)} tests.")

In [ ]:
# Results summary
print("\nGRID SEARCH RESULTS")
print("="*80)

# Filter to configs with signals
valid = grid_df[grid_df['signals'] > 0].copy()
print(f"Configs with signals: {len(valid)}/{len(grid_df)}")

# Beat baseline (62%)
beat_baseline = valid[valid['beat_rate'] > 0.62]
print(f"Configs beating 62% baseline: {len(beat_baseline)}/{len(valid)} ({len(beat_baseline)/len(valid)*100:.0f}%)")

# Beat 55%
beat_55 = valid[valid['beat_rate'] > 0.55]
print(f"Configs beating 55%: {len(beat_55)}/{len(valid)} ({len(beat_55)/len(valid)*100:.0f}%)")

# Best config
best = valid.loc[valid['beat_rate'].idxmax()]
print(f"\nBest config:")
print(f"  RL Z > {best['rl_z']}, MVRV > {best['mvrv_trigger']}, Trail {best['trailing_pct']*100:.0f}%")
print(f"  Beat rate: {best['beat_rate']*100:.0f}%")
print(f"  Avg excess: {best['avg_excess']*100:+.1f}%")

In [ ]:
# Top 10 configs
print("\nTOP 10 CONFIGURATIONS")
print("="*100)
print(f"{'RL Z':<8} {'MVRV':<8} {'Trail %':<10} {'Beat Rate':<12} {'Avg Excess':<12} {'Signals':<10}")
print("-"*100)

top10 = valid.nlargest(10, 'beat_rate')
for _, row in top10.iterrows():
    print(f"{row['rl_z']:<8} {row['mvrv_trigger']:<8} {row['trailing_pct']*100:<10.0f} "
          f"{row['beat_rate']*100:<11.0f}% {row['avg_excess']*100:<+11.1f}% {row['signals']:<10}")

In [ ]:
# Analyze by RL Z threshold
print("\nBEAT RATE BY RL Z THRESHOLD (averaged across exit params)")
print("="*60)
by_rl = valid.groupby('rl_z').agg({
    'beat_rate': 'mean',
    'signals': 'mean'
}).round(3)
print(by_rl)

In [ ]:
# Heatmap: RL Z vs MVRV trigger (averaged across trail %)
pivot = valid.pivot_table(
    values='beat_rate',
    index='rl_z',
    columns='mvrv_trigger',
    aggfunc='mean'
)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values * 100,
    x=[f"MVRV>{x}" for x in pivot.columns],
    y=[f"RL Z>{x}" for x in pivot.index],
    colorscale='RdYlGn',
    zmid=62,
    text=np.round(pivot.values * 100, 0),
    texttemplate="%{text}%",
    textfont={"size": 12}
))

fig.update_layout(
    title='Beat Rate: RL Z Threshold vs MVRV Exit Trigger<br>(Green = beats 62% baseline)',
    xaxis_title='MVRV Exit Trigger',
    yaxis_title='RL Z Entry Threshold',
    height=500
)
fig.show()

In [ ]:
# Compare to SOPR-only baseline
print("\n" + "="*80)
print("COMPARISON: SOPR+RL vs SOPR-only")
print("="*80)

# SOPR-only (RL Z = -inf, i.e., no RL filter)
def walk_forward_sopr_only(df, mvrv_trigger, trailing_pct):
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        # SOPR only - no RL filter
        sopr_signal = (test_df['sopr'] < 1) & (test_df['sopr_sth'] < 1)
        entries = sopr_signal & ~sopr_signal.shift(1).fillna(False)
        
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({'beat_hold': strat_return > hold_return})
    
    return pd.DataFrame(results)['beat_hold'].mean()

sopr_baseline = walk_forward_sopr_only(df_test, 2.25, 0.20)
print(f"SOPR-only (MVRV>2.25, 20% trail): {sopr_baseline*100:.0f}%")

# Best SOPR+RL
print(f"Best SOPR+RL config: {best['beat_rate']*100:.0f}%")
print(f"Improvement: {(best['beat_rate'] - sopr_baseline)*100:+.0f}%")

---
## Verdict

In [ ]:
print("\n" + "="*80)
print("ROBUSTNESS ANALYSIS VERDICT")
print("="*80)

configs_beating_baseline = len(beat_baseline)
total_valid = len(valid)
robustness_pct = configs_beating_baseline / total_valid * 100 if total_valid > 0 else 0

print(f"\n📊 ROBUSTNESS CHECK")
print(f"   Configs beating 62% baseline: {configs_beating_baseline}/{total_valid} ({robustness_pct:.0f}%)")

print(f"\n📊 BEST CONFIGURATION")
print(f"   Entry: SOPR < 1 AND STH SOPR < 1 AND RL Z > {best['rl_z']}")
print(f"   Exit: MVRV > {best['mvrv_trigger']} triggers {best['trailing_pct']*100:.0f}% trail")
print(f"   Beat rate: {best['beat_rate']*100:.0f}%")

print(f"\n🎯 VERDICT:")
if robustness_pct >= 50:
    print(f"   ✅ ROBUST! {robustness_pct:.0f}% of configs beat baseline.")
    print(f"   → Move STRAT-002 to PAPER-READY!")
elif robustness_pct >= 25:
    print(f"   ⚠️ PARTIALLY ROBUST. {robustness_pct:.0f}% of configs beat baseline.")
    print(f"   → Promising but needs more investigation.")
else:
    print(f"   ❌ NOT ROBUST. Only {robustness_pct:.0f}% of configs beat baseline.")
    print(f"   → Initial 67% was likely overfit. Stick with STRAT-001.")

print("\n" + "="*80)

In [ ]:
# Save results
import json

def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'strategy': 'sopr_realized_loss',
    'grid_results': to_native(grid_df.to_dict('records')),
    'total_configs': len(grid_df),
    'configs_with_signals': len(valid),
    'configs_beating_baseline': configs_beating_baseline,
    'robustness_pct': robustness_pct,
    'best_config': to_native(dict(best)),
    'baseline': {'strategy': 'sopr_only', 'beat_rate': sopr_baseline}
}

with open('../data/sopr_rl_robustness_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/sopr_rl_robustness_results.json")